# Sandbox — Silver Products

Exploração por trás de `silver_rules.tratar_products`. Foco: normalizar `product_category_name` (usada em joins com a tabela de tradução) e **castar as dimensões físicas** (peso e medidas para double, contagens para int).

> **Curiosidade:** o dataset Olist tem um typo de origem — as colunas se chamam `product_name_lenght` e `product_description_lenght` (o correto seria "length"). Mantemos o nome original de propósito, para não divergir da fonte.

In [ ]:
from utils.spark_utils import create_spark_session
from pyspark.sql.functions import col, trim, lower, current_timestamp

spark = create_spark_session('Sandbox-Silver-Products')

df_bronze = spark.read.parquet('s3a://bronze/olist/products')
df_bronze.show(5, truncate=False)
df_bronze.printSchema()

Conferindo as categorias existentes (serão normalizadas para minúsculo antes do join com `category_name`).

In [ ]:
df_bronze.select('product_category_name').distinct().show(10, truncate=False)

In [ ]:
df_silver = (df_bronze
    .withColumn('product_id', trim(col('product_id')))
    .withColumn('product_category_name', lower(trim(col('product_category_name'))))
    .withColumn('product_name_lenght', col('product_name_lenght').cast('int'))
    .withColumn('product_description_lenght', col('product_description_lenght').cast('int'))
    .withColumn('product_photos_qty', col('product_photos_qty').cast('int'))
    .withColumn('product_weight_g', col('product_weight_g').cast('double'))
    .withColumn('product_length_cm', col('product_length_cm').cast('double'))
    .withColumn('product_height_cm', col('product_height_cm').cast('double'))
    .withColumn('product_width_cm', col('product_width_cm').cast('double'))
    .withColumn('dt_criacao_silver', current_timestamp()))

df_silver.printSchema()
df_silver.show(5, truncate=False)

**Lógica promovida para `silver_rules.tratar_products`.**

In [ ]:
spark.stop()